# STEP 1: MODEL TRAINING

Cell 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

Cell 2: Load Dataset

In [2]:
df = pd.read_csv("English_learning_data.csv")
df.head()

,student_id,session_id,session_date,age,level,vocabulary_score,grammar_score,fluency_score,response_time,correct_answers,incorrect_answers,feedback_received,adaptive_content,overall_performance,engagement_score,learning_efficiency,learning_outcome
0,1,1,2025-01-01,24,Intermediate,69.663932,84.112928,45.183929,11.339841,24,2,Needs Improvement,0,66.320263,62.627267,0.744324,0
1,2,2,2025-01-02,21,Advanced,72.514709,73.547150,42.432883,19.934568,15,7,Excellent,0,62.831581,84.170261,0.828437,0
2,3,3,2025-01-03,28,Intermediate,63.963946,52.908663,69.050755,5.157405,28,3,Excellent,0,61.974455,80.075510,0.758712,0
3,4,4,2025-01-04,25,Intermediate,70.787101,73.730393,61.303549,18.425979,10,5,Needs Improvement,1,68.607014,81.973244,0.708355,0
4,5,5,2025-01-05,22,Intermediate,56.450872,54.836382,67.401618,7.237112,10,9,Excellent,1,59.562957,72.966396,0.941504,0


Cell 3: Data Cleaning


In [3]:
df = df.drop(['student_id', 'session_id', 'session_date'], axis=1)
df = df.dropna()

 Cell 4: Feature Engineering

In [4]:
# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Check available columns
print("Columns:", df.columns.tolist())

# If score column name is different, rename it manually
# Example: if marks exists instead of score
if 'score' not in df.columns:
    if 'marks' in df.columns:
        df.rename(columns={'marks':'score'}, inplace=True)
    elif 'result' in df.columns:
        df.rename(columns={'result':'score'}, inplace=True)

# Create features safely
if all(col in df.columns for col in ['score','time_spent','attempts']):
    df['efficiency'] = df['score'] / (df['time_spent'] + 1e-8)
    df['attempt_efficiency'] = df['score'] / (df['attempts'] + 1)

    print("Fixed Successfully")
    print(df.head())
else:
    print("score/time_spent/attempts column missing")

Columns: ['age', 'level', 'vocabulary_score', 'grammar_score', 'fluency_score', 'response_time', 'correct_answers', 'incorrect_answers', 'feedback_received', 'adaptive_content', 'overall_performance', 'engagement_score', 'learning_efficiency', 'learning_outcome']
score/time_spent/attempts column missing


Cell 5: Encoding

In [5]:
le1 = LabelEncoder()
le2 = LabelEncoder()

df['level'] = le1.fit_transform(df['level'])
df['feedback_received'] = le2.fit_transform(df['feedback_received'])

Cell 6: Scaling

In [6]:
from sklearn.preprocessing import StandardScaler
import pickle

# Clean column names
df.columns = df.columns.str.strip().str.lower()

print("Available columns:", df.columns.tolist())

# If score named differently, rename if needed
if 'marks' in df.columns:
    df.rename(columns={'marks':'score'}, inplace=True)

# Create missing columns if not present
if 'score' not in df.columns:
    df['score'] = 0

if 'time_spent' not in df.columns:
    df['time_spent'] = 1

if 'attempts' not in df.columns:
    df['attempts'] = 0

# Feature engineering
df['efficiency'] = df['score'] / (df['time_spent'] + 1e-8)
df['attempt_efficiency'] = df['score'] / (df['attempts'] + 1)

# Columns for scaling
cols = ['time_spent','score','attempts','efficiency','attempt_efficiency']

# StandardScaler fit + transform
scaler = StandardScaler()
df[cols] = scaler.fit_transform(df[cols])

# Save scaler
pickle.dump(scaler, open("scaler.pkl","wb"))

print("Scaler fitted successfully")
print(df.head())

Available columns: ['age', 'level', 'vocabulary_score', 'grammar_score', 'fluency_score', 'response_time', 'correct_answers', 'incorrect_answers', 'feedback_received', 'adaptive_content', 'overall_performance', 'engagement_score', 'learning_efficiency', 'learning_outcome']
Scaler fitted successfully
   age  level  vocabulary_score  grammar_score  fluency_score  response_time  \
0   24      2         69.663932      84.112928      45.183929      11.339841   
1   21      0         72.514709      73.547150      42.432883      19.934568   
2   28      2         63.963946      52.908663      69.050755       5.157405   
3   25      2         70.787101      73.730393      61.303549      18.425979   
4   22      2         56.450872      54.836382      67.401618       7.237112   

   correct_answers  incorrect_answers  feedback_received  adaptive_content  \
0               24                  2                  1                 0   
1               15                  7                  0      

Cell 7: Train Model

In [7]:
X = df.drop('learning_outcome', axis=1)
y = df['learning_outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Cell 8: Save Model

In [8]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

# STEP 2: AUTH + DB SETUP


 Cell 9: Create Users DB

In [9]:
import sqlite3

conn = sqlite3.connect("users.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE,
    password BLOB,
    role TEXT
)
""")
conn.commit()

Cell 10: Password Hashing


In [10]:
import bcrypt

def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt())

Cell 11: Register/Login

In [11]:
def register(username, password, role):
    hashed = hash_password(password)
    cursor.execute("INSERT INTO users (username, password, role) VALUES (?, ?, ?)",
                   (username, hashed, role))
    conn.commit()

def login(username, password):
    cursor.execute("SELECT * FROM users WHERE username=?", (username,))
    user = cursor.fetchone()
    
    if user and bcrypt.checkpw(password.encode(), user[2]):
        return user
    return None

# STEP 3: FLASK API (DEPLOYABLE BACKEND)

 Cell 12: API Setup

In [12]:
from flask import Flask, request, jsonify
import pickle
import numpy as np
import sqlite3

app = Flask(__name__)

model = pickle.load(open("model.pkl","rb"))
scaler = pickle.load(open("scaler.pkl","rb"))

conn = sqlite3.connect("students.db", check_same_thread=False)
cursor = conn.cursor()

Cell 13: Prediction API

In [13]:
@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    
    t = data['time_spent']
    s = data['score']
    a = data['attempts']

    eff = s/(t+1)
    att_eff = s/(a+1)

    X = [[t,s,a,0,1,eff,att_eff]]
    X = scaler.transform(X)

    pred = model.predict(X)[0]
    return jsonify({"prediction": int(pred)})

Cell 14: Chat API

In [14]:
import openai
openai.api_key = "YOUR_KEY"

@app.route('/chat', methods=['POST'])
def chat():
    q = request.json['question']

    res = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":q}]
    )

    return jsonify({"response": res['choices'][0]['message']['content']})

Cell 15: Run API

In [15]:
app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\dell\.conda\envs\tf_env\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# STEP 4: STREAMLIT DASHBOARD (FRONTEND)

Cell 16: UI Setup

In [16]:
import streamlit as st
import requests

st.title("🎓 Intelligent Tutoring System")

2026-07-08 20:22:52.573 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.418 
  command:

    streamlit run C:\Users\dell\.conda\envs\tf_env\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-08 20:22:54.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

Cell 17: Prediction UI

In [17]:
t = st.number_input("Time")
s = st.number_input("Score")
a = st.number_input("Attempts")

if st.button("Predict"):
    res = requests.post("http://127.0.0.1:5000/predict",
                        json={"time_spent":t,"score":s,"attempts":a})
    st.write(res.json())

2026-07-08 20:22:54.456 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.464 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.468 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.470 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.474 Session state does not function when running a script without `streamlit run`
2026-07-08 20:22:54.476 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.482 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22

Cell 18: Chatbot UI

In [18]:
q = st.text_input("Ask AI")

if st.button("Ask"):
    res = requests.post("http://127.0.0.1:5000/chat",
                        json={"question":q})
    st.write(res.json()["response"])

2026-07-08 20:22:54.553 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.556 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.557 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.560 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.562 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.564 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.566 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-08 20:22:54.570 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

# create app.py + api.py + deploy of the project

In [ ]:
# =========================================
# FINAL ADVANCED ITS (ONE CELL - STABLE)
# =========================================

import os, sys, subprocess, time, webbrowser

# 🔑 OPTIONAL (for real AI answers)
os.environ["OPENAI_API_KEY"] = ""

# INSTALL
os.system(f"{sys.executable} -m pip install flask bcrypt PyJWT streamlit requests pandas matplotlib openai sympy")

# ---------------- API ----------------
api_code = '''
from flask import Flask, request, jsonify
import sqlite3, datetime, bcrypt, os, re
import sympy as sp

try:
    import jwt
except:
    jwt=None

client=None
try:
    from openai import OpenAI
    if os.getenv("OPENAI_API_KEY"):
        client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
except:
    client=None

app=Flask(__name__)
SECRET_KEY="secret123"

conn=sqlite3.connect("its.db", check_same_thread=False)
cursor=conn.cursor()

cursor.execute("CREATE TABLE IF NOT EXISTS users(id INTEGER PRIMARY KEY, username TEXT UNIQUE, password BLOB)")
cursor.execute("CREATE TABLE IF NOT EXISTS scores(id INTEGER PRIMARY KEY, username TEXT, score REAL)")
cursor.execute("CREATE TABLE IF NOT EXISTS chats(id INTEGER PRIMARY KEY, username TEXT, question TEXT, answer TEXT)")
conn.commit()

@app.route('/register',methods=['POST'])
def register():
    d=request.json
    try:
        h=bcrypt.hashpw(d['password'].encode(),bcrypt.gensalt())
        cursor.execute("INSERT INTO users(username,password) VALUES(?,?)",(d['username'],h))
        conn.commit()
        return jsonify({"msg":"registered"})
    except:
        return jsonify({"error":"user exists"})

@app.route('/login',methods=['POST'])
def login():
    d=request.json
    cursor.execute("SELECT * FROM users WHERE username=?",(d['username'],))
    u=cursor.fetchone()

    if u and bcrypt.checkpw(d['password'].encode(),u[2]):
        token=jwt.encode({"user":d['username'],"exp":datetime.datetime.utcnow()+datetime.timedelta(hours=2)},
                         SECRET_KEY,algorithm="HS256") if jwt else "ok"
        return jsonify({"token":token})

    return jsonify({"error":"invalid"}),401

def verify(t):
    if not jwt: return True
    try:
        jwt.decode(t,SECRET_KEY,algorithms=["HS256"])
        return True
    except:
        return False

@app.route('/save',methods=['POST'])
def save():
    if not verify(request.headers.get("Authorization")):
        return jsonify({"error":"unauthorized"}),403

    d=request.json
    cursor.execute("INSERT INTO scores(username,score) VALUES(?,?)",(d['user'],d['score']))
    conn.commit()
    return jsonify({"msg":"saved"})

@app.route('/history',methods=['POST'])
def history():
    if not verify(request.headers.get("Authorization")):
        return jsonify({"error":"unauthorized"}),403

    u=request.json['user']
    cursor.execute("SELECT score FROM scores WHERE username=?",(u,))
    return jsonify({"scores":[r[0] for r in cursor.fetchall()]})

@app.route('/chat_history',methods=['POST'])
def chat_history():
    if not verify(request.headers.get("Authorization")):
        return jsonify({"error":"unauthorized"}),403

    u=request.json['user']
    cursor.execute("SELECT question,answer FROM chats WHERE username=?",(u,))
    return jsonify({"history":cursor.fetchall()})

# ---------- ADVANCED MATH ENGINE ----------
def math_engine(q):
    q=q.lower().strip().replace("^","**")

    try:
        if "=" in q:
            x=sp.symbols('x')
            left,right=q.split("=")
            eq=sp.Eq(sp.sympify(left),sp.sympify(right))
            return f"Solution: {sp.solve(eq,x)}"

        if "sqrt" in q:
            num=int(re.findall(r'\\d+',q)[0])
            return f"Square root = {sp.sqrt(num)}"

        nums=list(map(int,re.findall(r'\\d+',q)))

        if "add" in q or "sum" in q:
            return f"Result = {sum(nums)}"

        if "subtract" in q:
            return f"Result = {nums[1]-nums[0]}" if len(nums)>=2 else "Need 2 numbers"

        if "difference" in q:
            return f"Result = {nums[0]-nums[1]}" if len(nums)>=2 else "Need 2 numbers"

        if "multiply" in q:
            r=1
            for n in nums: r*=n
            return f"Result = {r}"

        if "divide" in q:
            return f"Result = {nums[0]/nums[1]}" if len(nums)>=2 and nums[1]!=0 else "Invalid division"

        return f"Result = {sp.sympify(q)}"

    except:
        return None

@app.route('/chat',methods=['POST'])
def chat():
    if not verify(request.headers.get("Authorization")):
        return jsonify({"error":"unauthorized"}),403

    user=request.json['user']
    q=request.json['question']

    ans=math_engine(q)

    if not ans:
        if client:
            try:
                res=client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role":"user","content":q}]
                )
                ans=res.choices[0].message.content
            except:
                ans="AI error"
        else:
            ans="Add API key for full AI answers."

    cursor.execute("INSERT INTO chats(username,question,answer) VALUES(?,?,?)",(user,q,ans))
    conn.commit()

    return jsonify({"response":ans})

app.run(port=5000)
'''

# ---------------- STREAMLIT ----------------
app_code = '''
import streamlit as st
import requests
import pandas as pd

st.set_page_config(layout="wide")
st.title("Advanced Intelligent Tutoring System")

if "token" not in st.session_state:
    st.session_state.token=None
    st.session_state.user=None

# LOGIN
if not st.session_state.token:
    u=st.text_input("Username")
    p=st.text_input("Password",type="password")

    if st.button("Register"):
        requests.post("http://127.0.0.1:5000/register",json={"username":u,"password":p})
        st.success("Registered")

    if st.button("Login"):
        r=requests.post("http://127.0.0.1:5000/login",json={"username":u,"password":p})
        if "token" in r.json():
            st.session_state.token=r.json()["token"]
            st.session_state.user=u
            st.rerun()
        else:
            st.error("Invalid login")

# MAIN
else:
    headers={"Authorization":st.session_state.token}
    menu=st.sidebar.radio("Menu",["Dashboard","Chatbot","Progress","History"])

    # DASHBOARD
    if menu=="Dashboard":
        st.subheader("Advanced Dashboard")

        col1,col2=st.columns(2)

        with col1:
            score=st.number_input("Score",0.0)
            study_hours=st.number_input("Study Hours",1.0)
            topic=st.selectbox("Topic",["Math","AI","ML","Python"])

        with col2:
            time_spent=st.number_input("Time Spent",10.0)
            attempts=st.number_input("Attempts",1)

        if st.button("Analyze"):
            requests.post("http://127.0.0.1:5000/save",
                          json={"user":st.session_state.user,"score":score},
                          headers=headers)

            perf=(score*0.5)+(study_hours*2)-(attempts*1.5)+(time_spent*0.2)

            level="Advanced" if perf>80 else "Intermediate" if perf>50 else "Beginner"

            weakness="Weak concepts" if score<40 else "Good"

            rec=f"Improve in {topic}"

            st.metric("Performance",round(perf,2))
            st.metric("Level",level)
            st.metric("Weakness",weakness)
            st.success("Recommendation: "+rec)

        st.bar_chart({"Study":[2,3,4,3,5,2,4]})

    # CHATBOT
    elif menu=="Chatbot":
        q=st.text_input("Ask anything")

        if st.button("Send"):
            r=requests.post("http://127.0.0.1:5000/chat",
                            json={"question":q,"user":st.session_state.user},
                            headers=headers)
            st.success(r.json()["response"])

    # PROGRESS
    elif menu=="Progress":
        r=requests.post("http://127.0.0.1:5000/history",
                        json={"user":st.session_state.user},headers=headers)
        scores=r.json().get("scores",[])
        if scores:
            df=pd.DataFrame({"Attempt":range(1,len(scores)+1),"Score":scores})
            st.line_chart(df.set_index("Attempt"))

    # HISTORY
    elif menu=="History":
        r=requests.post("http://127.0.0.1:5000/chat_history",
                        json={"user":st.session_state.user},headers=headers)
        for q,a in r.json().get("history",[]):
            st.write("Q:",q)
            st.write("A:",a)
            st.write("---")

    if st.sidebar.button("Logout"):
        st.session_state.token=None
        st.rerun()
'''

# SAVE FILES
open("api.py","w",encoding="utf-8").write(api_code)
open("app.py","w",encoding="utf-8").write(app_code)

# RUN
subprocess.Popen([sys.executable,"api.py"])
time.sleep(3)

webbrowser.open("http://localhost:8501")
os.system("streamlit run app.py --server.headless true")